[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okumura-kobe/kobe-data-business/blob/main/lecture05/lecture5_notebook.ipynb)

# 後期大学院演習「データで動かす社会とビジネス」
## 第5回　演習 Notebook

**テーマ：途上与信モデルを改訂するということ**
― 「より良いモデル」をどう考えるか ―

---

### この Notebook の使い方

- **STEP 1〜3**：コードを実行してデータを確認し、結果をワークシートに記録してください
- **STEP 4〜5**：コードで既存の履歴特徴量を比較・評価し、ワークシートの判断材料にしてください
- **第6回**では、教員が設定した Model B（Candidate Model）を全員で推定します

> ⚠️ **注意**：第5回では Logit 推定や AUC 計算はまだ行いません。それらは第6回以降です。

**今日の核心**
> 「より良いモデル」とは何か。
> AUC が高いモデルは、必ず適正与信を実現するのか。

---

| | |
|---|---|
| 学籍番号 | |
| 氏名 | |
| 実施日 | |

---
## Step 0　準備

In [ ]:
# ライブラリの読み込み（変更不要）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize']    = (11, 4)

print('Ready')

In [ ]:
# データの読み込み（変更不要）
# credit_base.csv：UCI Default of Credit Card Clients を母体にした教育用データセット（CC BY 4.0）
url = "https://raw.githubusercontent.com/okumura-kobe/kobe-data-business/main/data/credit_base.csv"
df = pd.read_csv(url)

print(f'件数: {len(df):,}行　変数数: {df.shape[1]}列')
print(f'Default件数: {df["default"].sum():,}件　Default率: {df["default"].mean():.4f}')
df.head()

---
## STEP 1　データを理解する（10分）

変数の意味・件数・欠損・Default率を確認します。
結果はワークシートの **STEP 1** に記録してください。

In [ ]:
# 1-1　基本情報の確認
print('=== 基本情報 ===')
print(f'件数: {len(df):,}行')
print(f'変数数: {df.shape[1]}列')
print(f'欠損値: {df.isnull().sum().sum()}件')
print(f'Default件数: {df["default"].sum():,}件')
print(f'Default率: {df["default"].mean():.4f}')

In [ ]:
# 1-2　変数の型と基本統計
print('=== データ型 ===')
print(df.dtypes)
print()
print('=== 基本統計量 ===')
df.describe()

---
## STEP 2　Default率を確認する（10分）

属性・与信情報別にDefault率を比較し、どの変数がDefaultと関係しているかを把握します。
結果はワークシートの **STEP 2** に記録してください。

> ※ 属性との統計的な関連が見られても、その属性を与信判断に利用することの妥当性は別途検討する必要があります。

In [ ]:
# 2-1　属性別 Default率
print('=== sex 別 Default率 ===')
print(df.groupby('sex')['default'].mean().round(4))

print()
print('=== education 別 Default率 ===')
print(df.groupby('education')['default'].mean().round(4))

print()
print('=== marriage 別 Default率 ===')
print(df.groupby('marriage')['default'].mean().round(4))

In [ ]:
# 2-2　与信情報別 Default率
print('=== latest_payment_status 別 Default率 ===')
print(df.groupby('latest_payment_status')['default'].agg(['mean','count']).round(4))

print()
# credit_limit を3分位に区切って比較
df['limit_group'] = pd.qcut(df['credit_limit'], q=3, labels=['低','中','高'])
print('=== credit_limit 別（低・中・高）Default率 ===')
print(df.groupby('limit_group', observed=True)['default'].mean().round(4))

print()
# credit_utilization を区間で区切って比較
df['util_group'] = pd.cut(df['credit_utilization'], bins=[0, 1, 3, 6], labels=['0〜1','1〜3','3〜'], include_lowest=True)
print('=== credit_utilization 別 Default率 ===')
print(df.groupby('util_group', observed=True)['default'].mean().round(4))

---
## STEP 3　返済履歴を確認する（10分）

履歴特徴量について、Default = 1 / Default = 0 別の平均を比較します。
どの変数がDefaultと強く関連しているかを把握し、ワークシートの **STEP 3** に記録してください。

In [ ]:
# 3-1　履歴特徴量の Default=0/1 別 平均比較
hist_vars = ['max_delay','months_with_delay','avg_bill_amount',
             'avg_payment_amount','credit_utilization','payment_ratio']

print('=== Default=1 vs Default=0 の平均比較 ===')
comparison = df.groupby('default')[hist_vars].mean().T
comparison.columns = ['Default=0（非Default）', 'Default=1（Default）']
comparison['差（1-0）'] = comparison['Default=1（Default）'] - comparison['Default=0（非Default）']
comparison['差の比率'] = (comparison['差（1-0）'] / comparison['Default=0（非Default）']).round(4)
print(comparison.round(4))

---
## STEP 4　Model Bへの候補変数を比較する（10分）

STEP 3の結果を踏まえ、各履歴特徴量が Model A にない情報をどれだけ持っているかを確認します。
コードの結果を参考に、ワークシートの **STEP 4** の表を埋めてください。

> **Model A（Current Model）の変数**：
> `credit_limit`, `age`, `latest_payment_status`, `latest_bill_amount`, `latest_payment_amount`

> ⚠️ **重要**：第5回では、ワークシートへの記録が主目的です。
> 「どの特徴量を追加すべきか」の判断訓練をしてください。
> 実際の Model B の仕様は教員が設定し、**第6回で全員共通のモデルを推定**します。

In [ ]:
# 4-1　Model Aの変数と履歴特徴量の相関（Defaultとの相関係数）
hist_vars = ['max_delay','months_with_delay','avg_bill_amount',
             'avg_payment_amount','credit_utilization','payment_ratio']

vars_A = ['credit_limit','age','latest_payment_status',
          'latest_bill_amount','latest_payment_amount']

print('=== Defaultとの相関係数 ===')
all_vars = vars_A + hist_vars
corr_with_default = df[all_vars + ['default']].corr()['default'].drop('default').sort_values(key=abs, ascending=False)
print(corr_with_default.round(4))

print()
print('=== 参考：Model Aの変数と履歴特徴量の相関（情報の重複確認） ===')
# max_delayとlatest_payment_statusの相関（情報の重複）
for hv in hist_vars:
    corr_a = df[['latest_payment_status', hv]].corr().iloc[0,1]
    print(f'  latest_payment_status × {hv}: {corr_a:.4f}')

---
## STEP 5　Model Bへの追加特徴量を整理する（10分）

**ワークシートの STEP 5** に、選択した特徴量と理由を記録してください。

以下の確認コードは、判断の参考として使用してください。

> 📝 **思考訓練として**：あなたが判断した追加特徴量の選択と理由をワークシートに記録してください。
> ただし第6回では、教員が設定した共通の Model B を全員で推定します。
> この判断は「モデル開発者としての思考訓練」です。

In [ ]:
# 5-1　候補特徴量の最終確認（判断の参考）
hist_vars = ['max_delay','months_with_delay','avg_bill_amount',
             'avg_payment_amount','credit_utilization','payment_ratio']

print('=== 各特徴量の確認 ===')
for v in hist_vars:
    mean_d1 = df[df['default']==1][v].mean()
    mean_d0 = df[df['default']==0][v].mean()
    corr = df[v].corr(df['default'])
    print(f'{v:<25}: Default=1の平均={mean_d1:.3f}  Default=0の平均={mean_d0:.3f}  Defaultとの相関={corr:.4f}')

In [ ]:
# 5-2　ワークシートへの記録（コメントとして記録してください）
#
# 【Model Bに追加する特徴量の選択（あなたの判断）】
#
# 追加する：
#
#
# 追加しない：
#
#
# 【選択の主な理由】
#
#
# 【懸念点・第6回以降で確認したいこと】
#
#


---
## まとめ

### 今日確認したこと

| STEP | 内容 |
|---|---|
| STEP 1 | credit_base.csv の変数・件数・Default率を確認した |
| STEP 2 | 属性・与信情報別の Default率を比較した |
| STEP 3 | 履歴特徴量の Default=0/1 別の平均を比較した |
| STEP 4 | Model A にない情報を履歴特徴量で評価した |
| STEP 5 | Model B に追加する特徴量を選択し、理由を整理した（思考訓練） |

### 今日の核心

> **「より良いモデル」とは、精度ではなく責任のことだ。**
> モデルを変えることは、顧客の与信管理を変えることだ。

---

### 第6回の予告

全員共通の **Model B（Candidate Model）** を使って PD を推定します。
- Model A と Model B を比較し、「本当に改善しているのか」を検証します
- 第5回の皆さんの判断と、実際の Model B を照らし合わせてください

**事前課題**
- 今日のワークシート STEP 5 の判断と理由を整理しておく
- BEEF+ の第6回配布資料を読む